# Experiment — Closed-Loop LLM Evaluation on Scenery (100 targets)

**Report section:** 7 — Conversational Layer (LLM round-trip evaluation)
**Goal:** Measure the cost of inserting a verbalizer LLM + parser LLM in front of GAUSSSEARCH's oracle. Target: lossless round-trip ($y_{\text{oracle}} \to \text{utterance} \to y_{\text{parsed}} = y_{\text{oracle}}$) with no query count penalty vs the pure Probit baseline.

**Setup:** Imports the production `comparison-search-app/app/` package directly — `SearchEngine`, `ParserLLM`, `VerbalizerLLM`, config — so the notebook evaluates exactly what the main app runs. 100 targets sampled from the 600-image scenery set (every 6th), in-query stopping, matched RNG per target.

**Stack:**
- **Parser:** `Nightshade2304/qwen3b-comparison-parser-v2` (Qwen 2.5-3B + QLoRA)
- **Verbalizer:** Qwen 2.5-1.5B-Instruct (prompted, no fine-tune)
- **Validation gate** (new since session 5): verbalizer regenerates up to 3 times if the parser doesn't recover the picked letter; falls back to the bare letter if all retries flip. Added to kill the ~10% positional-style flip rate from the previous closed-loop run.

**Experiments inside:**
1. Pure Probit baseline (floor)
2. Full closed-loop over 100 targets with the gated verbalizer
3. Per-style and per-class breakdown
4. Trajectory-level sanity: baseline and closed-loop should produce identical $y$-sequences since the gate is lossless
5. $\sigma_\epsilon$ noise decomposition — isolate algorithmic noise (Probit) from language noise (verbalizer+parser)

**Key results:**
- **Closed-loop mean = 15.1 queries, identical to pure Probit baseline** — zero language overhead (down from +26.9% in session 5's ungated run)
- **100% round-trip match across 1510 steps**, every style at 100% (was 87.7% aggregate, 50.7% on positional)
- $\sigma_\epsilon$ Probit noise: 1.1% flip rate against noiseless geometry. Verbalizer→parser: 0.0%. **All closed-loop noise is intentional algorithmic noise now.**

**Artifacts produced:** `closed_loop_100_v5_gated_fixed.pkl` — baseline + closed-loop results + full per-step logs.


## 1. Setup — load production stack

Imports the `comparison-search-app/app/` package so this notebook evaluates exactly what the main app runs — `SearchEngine`, `ParserLLM`, `VerbalizerLLM`, `parsed_to_y`, and the shared `config`.

Clearing `app.*` from `sys.modules` first guards against cached versions when iterating on the app code between runs.


In [1]:
import sys, gc, json, time, pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch

# Make the comparison-search-app package importable so we reuse
# the actual production code (SearchEngine, ParserLLM, VerbalizerLLM).
APP_ROOT = Path(r"C:\Users\shlok\projects\ddp-llm\comparison-search-app")
sys.path.insert(0, str(APP_ROOT))

# Force reload in case an older version is cached
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]

from app import config
from app.embeddings import Embeddings
from app.search_engine import SearchEngine
from app.parser_llm import ParserLLM, parsed_to_y
from app.verbalizer_llm import VerbalizerLLM

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)

print(f"config.SIGMA_EPS = {config.SIGMA_EPS}")
print(f"config.MAX_QUERIES = {config.MAX_QUERIES}")
print(f"parser adapter: {config.resolve_parser_adapter()}")

config.SIGMA_EPS = 0.05
config.MAX_QUERIES = 50
parser adapter: C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v2\checkpoint-final


## 2. Load embeddings + sample targets

Load the scenery embedding and sample 100 targets (every 6th of 600) with a per-target seed so each run is reproducible and independent. This matches the sampling in session 5's T-06 baseline so numbers are directly comparable.


In [2]:
emb = Embeddings()
print(f"loaded {emb.n} items, dim {emb.X.shape[1]}")
print(f"classes: {emb.classes()}")

# T-06 sampling: every 6th of the 600 → 100 targets
TARGET_INDICES = list(range(0, emb.n, 6))
print(f"sampling {len(TARGET_INDICES)} targets (every 6th)")

# per-target seed so runs are reproducible AND independent
BASE_SEED = 1000
def target_seed(target_idx):
    return BASE_SEED + target_idx

loaded 600 items, dim 10
classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
sampling 100 targets (every 6th)


## 3. Pure Probit baseline (floor)

Run GAUSSSEARCH with the oracle wired directly — no verbalizer, no parser. This is the floor: the number of queries GAUSSSEARCH takes when it hears the oracle perfectly every time. Any language stack added on top can only hurt (or at best tie).

Should reproduce the ~15.1 mean from session 5's T-06.


In [3]:
# Baseline: search consumes oracle answer directly. This is what GAUSSSEARCH
# is designed to run against. Sets the floor for what the closed-loop can do.
# Should reproduce ~15.1 mean queries from T-06.
print("=" * 60)
print("PURE PROBIT BASELINE")
print("=" * 60)

baseline_results = []
t_start = time.time()
for k, target_idx in enumerate(TARGET_INDICES):
    if k % 20 == 0:
        elapsed = time.time() - t_start
        print(f"  {k}/{len(TARGET_INDICES)}  ({elapsed:.1f}s elapsed)", flush=True)
    engine = SearchEngine(
        X=emb.X, sigma_eps=config.SIGMA_EPS,
        target_idx=int(target_idx),
        seed=target_seed(target_idx),
        max_queries=config.MAX_QUERIES,
    )
    while not engine.done:
        i, j = engine.propose_query()
        y = engine.oracle_answer(i, j)
        engine.apply_answer(y, status="clean")
    baseline_results.append({
        "target_idx": target_idx,
        "target_class": emb.label(target_idx),
        "steps": engine.step,
        "stop_reason": engine.stop_reason,
    })

t_baseline = time.time() - t_start
print(f"\ndone in {t_baseline:.1f}s")

steps = np.array([r["steps"] for r in baseline_results])
print(f"\nBaseline results ({len(steps)} targets):")
print(f"  mean:   {steps.mean():.1f}")
print(f"  median: {np.median(steps):.1f}")
print(f"  p95:    {np.percentile(steps, 95):.1f}")
print(f"  max:    {steps.max()}")

# per-class breakdown
per_class = defaultdict(list)
for r in baseline_results:
    per_class[r["target_class"]].append(r["steps"])
print(f"\nper class:")
for cls in sorted(per_class):
    arr = np.array(per_class[cls])
    print(f"  {cls:12s}  n={len(arr)}  mean={arr.mean():.1f}  median={np.median(arr):.1f}")

PURE PROBIT BASELINE
  0/100  (0.0s elapsed)
  20/100  (0.1s elapsed)
  40/100  (0.2s elapsed)
  60/100  (0.3s elapsed)
  80/100  (0.4s elapsed)

done in 0.5s

Baseline results (100 targets):
  mean:   15.1
  median: 15.0
  p95:    25.0
  max:    28

per class:
  buildings     n=17  mean=14.1  median=15.0
  forest        n=17  mean=17.5  median=17.0
  glacier       n=16  mean=15.2  median=13.0
  mountain      n=17  mean=13.6  median=14.0
  sea           n=17  mean=15.4  median=17.0
  street        n=16  mean=14.9  median=16.0


**Observation:**

| statistic | value |
|---|---|
| mean | **15.1** |
| median | 15.0 |
| p95 | 25.0 |
| max | 28 |

| class | n | mean | median |
|---|---|---|---|
| buildings | 17 | 14.1 | 15.0 |
| forest | 17 | 17.5 | 17.0 |
| glacier | 16 | 15.2 | 13.0 |
| mountain | 17 | 13.6 | 14.0 |
| sea | 17 | 15.4 | 17.0 |
| street | 16 | 14.9 | 16.0 |

Matches session 5's T-06 (15.1 mean) exactly. Baseline is the floor.


> **Note:** These 100 targets are a reproducible subsample of the 600 (every 6th, with per-target seeds). The full 600-target GAUSSSEARCH on scenery lives in `03_gauss_search_3d_viz.ipynb` and reports mean 16.9 queries — a separate experiment with its own RNG schedule. Both numbers are valid "GAUSSSEARCH on scenery"; the 15.1 here is the apples-to-apples floor for the closed-loop comparison in this notebook, nothing more.

## 4. Full closed-loop over 100 targets

The actual experiment: wrap each oracle answer in the verbalizer + parser round-trip, use the recovered label to drive ADF. The validation gate inside `VerbalizerLLM.verbalize()` regenerates up to 3 times with new seeds if the parser doesn't recover `[picked_letter]`, then falls back to the bare letter if all 3 retries flip.

Takes ~37 minutes on an RTX 5060 Laptop (parser + verbalizer both resident at 5.3 GB VRAM).


In [7]:
print("=" * 60)
print("CLOSED-LOOP FULL RUN (100 targets, in_query stopping)")
print("=" * 60)

print("loading parser...")
parser = ParserLLM()
free_gpu()
print("loading verbalizer...")
verbalizer = VerbalizerLLM()
free_gpu()

closed_loop_results = []
per_style_stats = defaultdict(lambda: {"match": 0, "skip": 0, "flip": 0, "fell_back": 0})

t_start = time.time()
for k, target_idx in enumerate(TARGET_INDICES):
    if k % 5 == 0:
        elapsed = time.time() - t_start
        eta = elapsed / (k + 1) * (len(TARGET_INDICES) - k - 1) if k > 0 else 0
        print(f"  {k}/{len(TARGET_INDICES)}  ({elapsed:.1f}s elapsed, ETA {eta:.0f}s)", flush=True)

    seed = target_seed(target_idx)
    engine = SearchEngine(
        X=emb.X, sigma_eps=config.SIGMA_EPS,
        target_idx=int(target_idx), seed=seed,
        max_queries=config.MAX_QUERIES,
    )
    rng_lang = np.random.default_rng(seed + 999_999)

    step_log = []
    while not engine.done:
        i, j = engine.propose_query()
        y_oracle = engine.oracle_answer(i, j)
        picked = "A" if y_oracle == 0 else "B"

        utt, style = verbalizer.verbalize(picked, rng_lang, parser=parser)
        parsed, _ = parser.parse(utt, options=("A", "B"))
        y_recovered, y_status = parsed_to_y(parsed)

        base_style = style.replace(" (fallback)", "")
        fell_back = "(fallback)" in style
        if fell_back:
            per_style_stats[base_style]["fell_back"] += 1

        if y_recovered is None:
            status = "skip"
            per_style_stats[base_style]["skip"] += 1
        elif y_recovered == y_oracle:
            status = "match"
            per_style_stats[base_style]["match"] += 1
        else:
            status = "flip"
            per_style_stats[base_style]["flip"] += 1

        if status == "skip":
            engine.apply_answer(None, status="skip")
        else:
            engine.apply_answer(y_recovered, status=status)

        step_log.append({
            "step": engine.step, "picked": picked, "utt": utt,
            "style": style, "status": status, "y_oracle": y_oracle,
            "y_recovered": y_recovered,
        })

    closed_loop_results.append({
        "target_idx": target_idx,
        "target_class": emb.label(target_idx),
        "steps": engine.step,
        "stop_reason": engine.stop_reason,
        "step_log": step_log,
    })

t_closed = time.time() - t_start
print(f"\ndone in {t_closed:.1f}s ({t_closed/60:.1f} min)")

with open(Path(r"C:\Users\shlok\projects\ddp-llm\scenery-search\notebooks") /
           "closed_loop_100_v5_gated_fixed.pkl", "wb") as f:
    pickle.dump({
        "baseline_results": baseline_results,
        "closed_loop_results": closed_loop_results,
        "per_style_stats": dict(per_style_stats),
    }, f)
print("saved to closed_loop_100_v5_gated_fixed.pkl")

del verbalizer, parser
free_gpu()

CLOSED-LOOP FULL RUN (100 targets, in_query stopping)
loading parser...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM: 2.19 GB
loading verbalizer...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM: 5.28 GB
  0/100  (0.0s elapsed, ETA 0s)
  5/100  (128.6s elapsed, ETA 2015s)
  10/100  (234.7s elapsed, ETA 1899s)
  15/100  (329.1s elapsed, ETA 1728s)
  20/100  (431.1s elapsed, ETA 1622s)
  25/100  (593.0s elapsed, ETA 1688s)
  30/100  (711.0s elapsed, ETA 1582s)
  35/100  (842.9s elapsed, ETA 1498s)
  40/100  (967.7s elapsed, ETA 1393s)
  45/100  (1043.1s elapsed, ETA 1225s)
  50/100  (1168.8s elapsed, ETA 1123s)
  55/100  (1262.1s elapsed, ETA 992s)
  60/100  (1378.6s elapsed, ETA 881s)
  65/100  (1480.2s elapsed, ETA 763s)
  70/100  (1594.3s elapsed, ETA 651s)
  75/100  (1732.9s elapsed, ETA 547s)
  80/100  (1807.2s elapsed, ETA 424s)
  85/100  (1942.2s elapsed, ETA 316s)
  90/100  (2062.2s elapsed, ETA 204s)
  95/100  (2172.8s elapsed, ETA 91s)

done in 2269.5s (37.8 min)
saved to closed_loop_100_v5_gated_fixed.pkl
VRAM: 0.01 GB


**Observation:** full 100-target run completes in 37.8 minutes. Pickle saved to `closed_loop_100_v5_gated_fixed.pkl` with `baseline_results`, `closed_loop_results` (including per-step `step_log`), and `per_style_stats`. All summary stats computed in Section 5.


## 5. Summary — baseline vs closed-loop, round-trip fidelity, per-style, per-class


In [8]:
# Summary stats + comparison against T-06 baseline (from session 5).
print("=" * 70)
print("CLOSED-LOOP RESULTS (100 targets, in_query stopping)")
print("=" * 70)

steps_baseline = np.array([r["steps"] for r in baseline_results])
steps_closed = np.array([r["steps"] for r in closed_loop_results])

print(f"\n{'metric':<15s} {'baseline':<12s} {'closed-loop':<14s} {'delta':<10s}")
print("-" * 55)
for name, fn in [("mean", np.mean), ("median", np.median),
                  ("p95", lambda x: np.percentile(x, 95)),
                  ("max", np.max)]:
    b = fn(steps_baseline)
    c = fn(steps_closed)
    print(f"{name:<15s} {b:<12.1f} {c:<14.1f} {c-b:+.1f}")

# Round-trip fidelity
total_steps = sum(len(r["step_log"]) for r in closed_loop_results)
n_match = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "match")
n_flip = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "flip")
n_skip = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "skip")

print(f"\nround-trip fidelity across all {total_steps} steps:")
print(f"  match:  {n_match:4d}  ({100*n_match/total_steps:.1f}%)")
print(f"  flip:   {n_flip:4d}  ({100*n_flip/total_steps:.1f}%)")
print(f"  skip:   {n_skip:4d}  ({100*n_skip/total_steps:.1f}%)")

# Per-style stats
print(f"\nper-style breakdown:")
print(f"  {'style':<12s} {'match':<8s} {'flip':<8s} {'skip':<8s} {'fell_back':<10s} {'match%':<8s}")
print("-" * 60)
for style_name in ["letter", "ordinal", "positional", "direct", "casual"]:
    if style_name not in per_style_stats:
        continue
    s = per_style_stats[style_name]
    n = s["match"] + s["flip"] + s["skip"]
    match_pct = 100 * s["match"] / n if n > 0 else 0
    print(f"  {style_name:<12s} {s['match']:<8d} {s['flip']:<8d} {s['skip']:<8d} {s['fell_back']:<10d} {match_pct:.1f}%")

# Per-class
per_class_b = defaultdict(list)
per_class_c = defaultdict(list)
for r in baseline_results:
    per_class_b[r["target_class"]].append(r["steps"])
for r in closed_loop_results:
    per_class_c[r["target_class"]].append(r["steps"])

print(f"\nper-class comparison:")
print(f"  {'class':<12s} {'baseline':<12s} {'closed':<12s} {'delta':<10s}")
print("-" * 50)
for cls in sorted(per_class_b):
    b = np.mean(per_class_b[cls])
    c = np.mean(per_class_c[cls])
    print(f"  {cls:<12s} {b:<12.1f} {c:<12.1f} {c-b:+.1f}")

# vs old T-06 numbers (from session 5 handover)
print(f"\n\n=== NEW STACK vs OLD T-06 RESULTS ===")
print(f"                  {'T-06 old':<12s} {'new':<12s} {'delta':<10s}")
print(f"baseline mean:    {'15.1':<12s} {steps_baseline.mean():<12.1f} {'(sanity)':<10s}")
print(f"closed-loop mean: {'19.2':<12s} {steps_closed.mean():<12.1f} {steps_closed.mean() - 19.2:+.1f}")
print(f"round-trip match: {'87.7%':<12s} {100*n_match/total_steps:<11.1f}% {100*n_match/total_steps - 87.7:+.1f}pp")
print(f"round-trip flip:  {'~10%':<12s} {100*n_flip/total_steps:<11.1f}%")
print(f"round-trip skip:  {'~2%':<12s} {100*n_skip/total_steps:<11.1f}%")

CLOSED-LOOP RESULTS (100 targets, in_query stopping)

metric          baseline     closed-loop    delta     
-------------------------------------------------------
mean            15.1         15.1           +0.0
median          15.0         15.0           +0.0
p95             25.0         25.0           +0.0
max             28.0         28.0           +0.0

round-trip fidelity across all 1510 steps:
  match:  1510  (100.0%)
  flip:      0  (0.0%)
  skip:      0  (0.0%)

per-style breakdown:
  style        match    flip     skip     fell_back  match%  
------------------------------------------------------------
  letter       296      0        0        9          100.0%
  ordinal      321      0        0        0          100.0%
  positional   282      0        0        0          100.0%
  direct       294      0        0        0          100.0%
  casual       317      0        0        0          100.0%

per-class comparison:
  class        baseline     closed       delta     
----

**Observation:**

| metric | baseline | closed-loop | delta |
|---|---|---|---|
| mean | 15.1 | **15.1** | **+0.0** |
| median | 15.0 | 15.0 | +0.0 |
| p95 | 25.0 | 25.0 | +0.0 |
| max | 28 | 28 | +0.0 |

**Round-trip fidelity across all 1510 steps:**
- match: **1510 / 1510 (100.0%)**
- flip: 0 (0.0%)
- skip: 0 (0.0%)

**Per-style:**

| style | match | flip | skip | fell back | match % |
|---|---|---|---|---|---|
| letter | 296 | 0 | 0 | 9 | 100.0% |
| ordinal | 321 | 0 | 0 | 0 | 100.0% |
| positional | 282 | 0 | 0 | 0 | 100.0% |
| direct | 294 | 0 | 0 | 0 | 100.0% |
| casual | 317 | 0 | 0 | 0 | 100.0% |

Every style at 100% match. Positional (which was 50.7% in the ungated run) is now 100% because the gate catches "the right one" → `[A]` flips and regenerates until a style-consistent utterance comes out. Letter style falls back 9 times — these are cases where the verbalizer 1.5B still produced something borderline-parseable on the first try that the gate rejected, then settled on the deterministic bare-letter fallback.

**vs session 5 T-06:**

| | T-06 (ungated) | now (gated) | delta |
|---|---|---|---|
| closed-loop mean | 19.2 | **15.1** | **−4.1 queries** |
| round-trip match | 87.7% | **100.0%** | **+12.3 pp** |
| round-trip flip | ~10% | 0.0% | — |
| round-trip skip | ~2% | 0.0% | — |

Zero language overhead and lossless round-trip. The gate makes the verbalizer→parser pipeline effectively invisible to the search algorithm.


## 6. Trajectory sanity — baseline vs closed-loop produce identical $y$-sequences

If the verbalizer→parser round-trip is truly lossless, then for a given target with matched RNG, the baseline and closed-loop runs should walk through the *same* query pairs with the *same* answers. Compare the first target's $y$-sequence between the two runs.


In [9]:
# Compare first target's (i, j) trajectory in baseline vs closed loop.
# If truly identical: verbalizer/parser are a clean pipe. If not: something
# in the closed loop is drifting the search engine's RNG.

target_idx = TARGET_INDICES[0]
seed = target_seed(target_idx)

# Baseline trajectory
engine_b = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                        target_idx=int(target_idx), seed=seed,
                        max_queries=config.MAX_QUERIES)
baseline_trajectory = []
while not engine_b.done:
    i, j = engine_b.propose_query()
    y = engine_b.oracle_answer(i, j)
    baseline_trajectory.append((engine_b.step, i, j, y))
    engine_b.apply_answer(y, status="clean")

# Closed-loop trajectory from the saved run
closed_trajectory = [(s["step"] - 1, None, None, s["y_oracle"])
                     for s in closed_loop_results[0]["step_log"]]
# We don't have (i, j) saved from closed loop, so compare y-sequences
print(f"target_idx={target_idx}")
print(f"baseline y-sequence:    {[t[3] for t in baseline_trajectory]}")
print(f"closed-loop y-sequence: {[t[3] for t in closed_trajectory]}")
print(f"match: {[t[3] for t in baseline_trajectory] == [t[3] for t in closed_trajectory]}")

# Show utterances and picked letters side-by-side
print(f"\nclosed-loop utterances:")
for s in closed_loop_results[0]["step_log"]:
    print(f"  step={s['step']}  picked={s['picked']}  y_oracle={s['y_oracle']}  utt={s['utt']!r}  status={s['status']}")

target_idx=0
baseline y-sequence:    [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
closed-loop y-sequence: [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
match: True

closed-loop utterances:
  step=1  picked=A  y_oracle=0  utt='A for sure.'  status=match
  step=2  picked=A  y_oracle=0  utt='A.'  status=match
  step=3  picked=B  y_oracle=1  utt='the right one'  status=match
  step=4  picked=A  y_oracle=0  utt='Got it! First one.'  status=match
  step=5  picked=B  y_oracle=1  utt='the right one'  status=match
  step=6  picked=B  y_oracle=1  utt='B it is.'  status=match
  step=7  picked=B  y_oracle=1  utt='the second one'  status=match
  step=8  picked=A  y_oracle=0  utt='A'  status=match
  step=9  picked=B  y_oracle=1  utt='the second one'  status=match
  step=10  picked=A  y_oracle=0  utt='I’ll go with A.'  status=match
  step=11  picked=B  y_oracle=1  utt='the second one'  status=match
  step=12  picked=A  y_oracle=0  utt='Right choice, A!'  status=match


**Observation:** both sequences are identical bit-for-bit (`[0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]`). The language pipeline doesn't shift the search's RNG trajectory, meaning the verbalizer's internal `torch.manual_seed(...)` calls stay isolated from the engine's `rng`. This is what makes the closed-loop mean exactly match baseline, not just approximately.

The sample utterances show the full style rotation at work: `"A for sure."`, `"the right one"`, `"Got it! First one."`, `"B it is."`, `"the second one"`, `"I'll go with A."`, `"Right choice, A!"` — every one correctly round-tripped.


## 7. $\sigma_\epsilon$ noise decomposition — isolate algorithmic noise from language noise

With the language stack at 0.0% flip, any disagreement between the oracle's answer and the noiseless geometric answer (which side of the bisecting hyperplane the target actually sits on) is pure $\sigma_\epsilon$ Probit noise. Measure it by re-running each search in a fresh engine and comparing `y_oracle` to the signed-distance geometry at each query.


In [13]:
from app.search_engine import bisecting_hyperplane

print("=" * 60)
print("σ_ε NOISE — noisy oracle vs noiseless geometry")
print("=" * 60)

n_agree = 0
n_disagree = 0
per_target_disagree = defaultdict(int)

for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]

    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)

    while not engine.done:
        i, j = engine.propose_query()
        x_i, x_j = emb.X[i], emb.X[j]
        w, b = bisecting_hyperplane(x_i, x_j)
        signed_dist = np.dot(x_t, w) + b
        y_noiseless = 0 if signed_dist > 0 else 1

        # Call oracle_answer to consume RNG the same way the real run did.
        y_noisy = engine.oracle_answer(i, j)

        if y_noiseless == y_noisy:
            n_agree += 1
        else:
            n_disagree += 1
            per_target_disagree[target_idx] += 1

        engine.apply_answer(y_noisy, status="clean")

total = n_agree + n_disagree
print(f"\ntotal steps: {total}")
print(f"  agree with noiseless:    {n_agree}  ({100*n_agree/total:.1f}%)")
print(f"  disagree (σ_ε flip):     {n_disagree}  ({100*n_disagree/total:.1f}%)")

flip_counts = [per_target_disagree.get(r["target_idx"], 0) for r in closed_loop_results]
print(f"\nper-target flip distribution:")
print(f"  mean flips per target:   {np.mean(flip_counts):.2f}")
print(f"  targets with 0 flips:    {sum(1 for f in flip_counts if f == 0)}/100")
print(f"  targets with 1 flip:     {sum(1 for f in flip_counts if f == 1)}/100")
print(f"  targets with 2+ flips:   {sum(1 for f in flip_counts if f >= 2)}/100")
print(f"  max flips on one target: {max(flip_counts)}")

# Also compute how confident the oracle is on average (helps interpret)
# by looking at probit_prob at each query.
from app.search_engine import probit_prob

confidences = []
for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]
    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)
    while not engine.done:
        i, j = engine.propose_query()
        p = probit_prob(emb.X[i], emb.X[j], x_t, config.SIGMA_EPS)
        confidences.append(max(p, 1 - p))  # confidence in the more likely answer
        y = engine.oracle_answer(i, j)
        engine.apply_answer(y, status="clean")

confidences = np.array(confidences)
print(f"\noracle confidence per query:")
print(f"  mean:              {confidences.mean():.3f}")
print(f"  median:            {np.median(confidences):.3f}")
print(f"  fraction >0.9:     {(confidences > 0.9).mean():.2f}")
print(f"  fraction <0.6:     {(confidences < 0.6).mean():.2f}  (near-coinflip queries)")

print(f"\n=== NOISE DECOMPOSITION ===")
print(f"σ_ε Probit oracle:     {100*n_disagree/total:.1f}% flip vs noiseless geometry")
print(f"Verbalizer→parser:      0.0% flip  (validation gate is lossless)")

σ_ε NOISE — noisy oracle vs noiseless geometry

total steps: 1510
  agree with noiseless:    1494  (98.9%)
  disagree (σ_ε flip):     16  (1.1%)

per-target flip distribution:
  mean flips per target:   0.16
  targets with 0 flips:    86/100
  targets with 1 flip:     12/100
  targets with 2+ flips:   2/100
  max flips on one target: 2

oracle confidence per query:
  mean:              0.987
  median:            1.000
  fraction >0.9:     0.96
  fraction <0.6:     0.01  (near-coinflip queries)

=== NOISE DECOMPOSITION ===
σ_ε Probit oracle:     1.1% flip vs noiseless geometry
Verbalizer→parser:      0.0% flip  (validation gate is lossless)


**Observation:**

| source | flip rate |
|---|---|
| $\sigma_\epsilon$ Probit oracle (vs noiseless geometry) | **1.1%** |
| Verbalizer → parser (gated) | **0.0%** |
| **Total closed-loop noise** | **1.1%** |

**Per-target flip distribution:**
- 86 / 100 targets: zero flips
- 12 / 100 targets: 1 flip
- 2 / 100 targets: 2 flips
- max per target: 2

**Oracle confidence (per query, under Probit at $\sigma_\epsilon = 0.05$):**
- mean $\max(p, 1-p) = 0.987$
- 96% of queries have confidence > 0.9
- only 1% have confidence < 0.6 (near-coinflip)

The 1.1% residual Probit noise is intentional (algorithm-designed) — not a defect to fix. It's lower than the 8-11% calibration target because scenery targets at SIGMA_EPS=0.05 are mostly comfortably far from the hyperplane. The whole closed-loop pipeline is now dominated entirely by this intentional noise; language is lossless.


## Summary

- **Closed-loop mean 15.1 = pure Probit mean 15.1** on 100 scenery targets. Zero language overhead.
- **100% round-trip match across 1510 steps** — every style (letter, ordinal, positional, direct, casual) at 100%.
- Fallback rate: 9/1510 steps (0.6%), all on the `letter` style, settled with the deterministic bare-letter fallback.
- $\sigma_\epsilon$ Probit noise: 1.1% flip rate against noiseless geometry. Verbalizer→parser: 0.0%.

The validation gate inside `VerbalizerLLM.verbalize()` eliminated the ~10% language flip rate that session 5's ungated run reported. The main app now has an LLM-driven interface that costs nothing in query count vs the direct-oracle baseline.

**What this unblocks for the report:** Section 7 (Conversational Layer) can quote 15.1 queries end-to-end with 100% round-trip and compare against session 5's T-06 (19.2 queries, 87.7% match) as the before/after of the gate.
